# 12 — Batch Pipeline

**Marker:** `NOTEBOOK_12_BATCH_PIPELINE_FRESH_V1`

This notebook ties the working project together.

It starts with the knowledge tree from Notebook 02, generates topic candidates,
lets you choose which candidates to run, and then orchestrates:

1. outline generation
2. script generation
3. script editing
4. fact checking
5. metadata
6. Kokoro narration
7. captions
8. final video assembly

The default first run processes **one selected topic** and reuses your single
Subway Surfers recording.


## Load the project

In [1]:
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "educational_shorts").is_dir():
            return candidate

    raise FileNotFoundError(
        "Could not find the project root containing educational_shorts/."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from educational_shorts.batch import (
    BatchConfig,
    preflight_batch_pipeline,
    prepare_topic_batch,
    run_batch_pipeline,
    summarize_batch_manifest,
    summarize_topic_plan,
)

print("NOTEBOOK_12_BATCH_PIPELINE_FRESH_V1")
print(f"Project root: {PROJECT_ROOT}")

ImportError: cannot import name 'fact_check_script' from 'educational_shorts.fact_checker' (c:\Users\hitch\python_files\educational_shorts\educational_shorts\fact_checker.py)

## Configuration

For the first test, keep `SELECTED_TOPIC_INDEXES = [0]`.

`PAUSE_ON_MANUAL_REVIEW = True` prevents a questionable script from
automatically becoming a public-facing video. It still saves the checked script
and fact-check report for inspection.


In [ ]:
ROOT_CATEGORY = "Science"
TREE_FILENAME = None

# Set to a full path such as ["Science", "Biology", "Microbiology"]
# to bypass random category selection. Leave as None for random selection.
CATEGORY_PATH_OVERRIDE = None

TOPIC_CANDIDATE_COUNT = 10
MIN_CATEGORY_DEPTH = 2
MAX_CATEGORY_DEPTH = 3

CATEGORY_SELECTION_SEED = 42
TOPIC_GENERATION_SEED = 42

# Choose candidate indexes after previewing the next section.
SELECTED_TOPIC_INDEXES = [0]

PAUSE_ON_MANUAL_REVIEW = True
CONTINUE_ON_ERROR = False
SKIP_EXISTING_FINAL = True

GAMEPLAY_SUBDIRECTORY = "subway_surfers"
BACKGROUND_VIDEO_FILENAME = None

TTS_VOICE = "am_michael"
TTS_SPEED = 1.0

CAPTION_STYLE = "phrase"
ASS_FONT_NAME = "Arial"
ASS_FONT_SIZE = 72
ASS_MARGIN_V = 300

OUTPUT_WIDTH = 1080
OUTPUT_HEIGHT = 1920
CROP_ANCHOR_Y = "top"
LOOP_BACKGROUND = True

config = BatchConfig(
    root_category=ROOT_CATEGORY,
    tree_filename=TREE_FILENAME,
    category_path_override=CATEGORY_PATH_OVERRIDE,
    topic_candidate_count=TOPIC_CANDIDATE_COUNT,
    min_category_depth=MIN_CATEGORY_DEPTH,
    max_category_depth=MAX_CATEGORY_DEPTH,
    category_selection_seed=CATEGORY_SELECTION_SEED,
    topic_generation_seed=TOPIC_GENERATION_SEED,
    pause_on_manual_review=PAUSE_ON_MANUAL_REVIEW,
    gameplay_subdirectory=GAMEPLAY_SUBDIRECTORY,
    background_video_filename=BACKGROUND_VIDEO_FILENAME,
    tts_voice=TTS_VOICE,
    tts_speed=TTS_SPEED,
    caption_style=CAPTION_STYLE,
    ass_font_name=ASS_FONT_NAME,
    ass_font_size=ASS_FONT_SIZE,
    ass_margin_v=ASS_MARGIN_V,
    output_width=OUTPUT_WIDTH,
    output_height=OUTPUT_HEIGHT,
    crop_anchor_y=CROP_ANCHOR_Y,
    loop_background=LOOP_BACKGROUND,
    skip_existing_final=SKIP_EXISTING_FINAL,
    continue_on_error=CONTINUE_ON_ERROR,
)

print(config.model_dump_json(indent=2))

## Preflight check

In [ ]:
preflight = preflight_batch_pipeline(
    project_root=PROJECT_ROOT,
    config=config,
)

for name, value in preflight.items():
    print(f"{name}: {value}")

## Generate topic candidates

This only generates and saves candidate topics. It does **not** yet create
scripts, audio, captions, or videos.


In [ ]:
topic_plan = prepare_topic_batch(
    project_root=PROJECT_ROOT,
    config=config,
)

for name, value in summarize_topic_plan(topic_plan).items():
    print(f"{name}: {value}")

## Preview candidates

In [ ]:
print("CATEGORY:")
print(" > ".join(topic_plan.category_path))
print()

for index, topic in enumerate(topic_plan.topics.topics):
    print(f"[{index}] {topic.title}")
    print(f"    {topic.learning_objective}")
    print()

## Confirm selected indexes

Edit `SELECTED_TOPIC_INDEXES` in the configuration cell, rerun that cell, and
then run this cell.

For the first complete test, one index is enough:

```python
SELECTED_TOPIC_INDEXES = [0]
```


In [ ]:
print(f"Selected indexes: {SELECTED_TOPIC_INDEXES}")
print()

for index in SELECTED_TOPIC_INDEXES:
    topic = topic_plan.topics.topics[index]
    print(f"[{index}] {topic.title}")

## Run the selected topics

This is the long-running cell. It may make multiple local Ollama calls, run
Kokoro, create captions, and invoke FFmpeg.

The Kokoro synthesizer is loaded once and reused when several topics are
selected.


In [ ]:
batch_manifest = run_batch_pipeline(
    project_root=PROJECT_ROOT,
    plan=topic_plan,
    selected_topic_indexes=SELECTED_TOPIC_INDEXES,
    config=config,
)

for name, value in summarize_batch_manifest(
    batch_manifest
).items():
    print(f"{name}: {value}")

## Review each topic result

In [ ]:
for item in batch_manifest.items:
    print("=" * 72)
    print(f"{item.item_number}. {item.topic_title}")
    print(f"Status: {item.status}")
    print(f"Final stage: {item.final_stage}")
    print(f"Message: {item.message}")
    print(
        "Fact-check verdict: "
        f"{item.fact_check_verdict}"
    )
    print(
        "Requires manual review: "
        f"{item.requires_manual_review}"
    )
    print("Completed stages:")
    for stage in item.stages_completed:
        print(f"  - {stage}")

    print("Paths:")
    for name, path in item.paths.items():
        print(f"  {name}: {path}")
    print()

## Preview completed videos

In [ ]:
from IPython.display import Video, display

completed_items = [
    item
    for item in batch_manifest.items
    if item.status in {"completed", "skipped_existing"}
    and item.paths.get("final_video")
]

if not completed_items:
    print(
        "No completed videos to preview. Check whether a topic was held "
        "for manual review or failed at an earlier stage."
    )
else:
    for item in completed_items:
        print(item.topic_title)
        display(
            Video(
                filename=item.paths["final_video"],
                embed=True,
                width=360,
            )
        )

## Inspect the saved batch manifest

In [ ]:
batch_manifest_path = (
    Path(batch_manifest.output_directory)
    / batch_manifest.manifest_filename
)

print(f"Saved manifest: {batch_manifest_path}")
print()
print(
    batch_manifest_path.read_text(
        encoding="utf-8"
    )
)